In [ ]:
import os
import requests
import sqlite3
from dotenv import load_dotenv

In [ ]:
con = sqlite3.connect("episodes.db")
cur = con.cursor()

In [ ]:
cur.execute("CREATE TABLE episodes(" \
"id INTEGER PRIMARY KEY, name, description, release_date, duration_ms, explicit, spotify_id)")

In [ ]:
res = cur.execute("Select name from sqlite_master")
res.fetchone()

In [ ]:
load_dotenv()
access_token = os.getenv("access_token")
                 
response = requests.request("GET", 
                 url="https://api.spotify.com/v1/shows/7BTOsF2boKmlYr76BelijW/episodes?offset=0&limit=50", 
                 headers={"Authorization": f"Bearer {access_token}"}
                 ).json()

cur.executemany("Insert into episodes (name, description, release_date, duration_ms, explicit, spotify_id)" \
                "values (?, ?, ?, ?, ?, ?)",
                [(item["name"], item["description"], item["release_date"], item["duration_ms"], item["explicit"], item["id"]) for item in response["items"]])


while response["next"]:

    response = requests.request("GET", url=response["next"], headers={"Authorization": f"Bearer {access_token}"}).json()

    cur.executemany("Insert into episodes (name, description, release_date, duration_ms, explicit, spotify_id)" \
                "values (?, ?, ?, ?, ?, ?)",
                [(item["name"], item["description"], item["release_date"], item["duration_ms"], item["explicit"], item["id"]) 
                 for item in response["items"]])
    

con.commit()



In [ ]:
row1 = con.execute("select count(*) from episodes")
row1.fetchone()

In [ ]:
con.close()